In [0]:

#read from dimensions
dim_customers = spark.table("ecommerce.e_comm_gold.dimcustomers")
dim_products = spark.table("ecommerce.e_comm_gold.dimproducts")

# craete temp views 
dim_customers.createOrReplaceTempView("vw_dim_customers")
dim_products.createOrReplaceTempView("vw_dim_products")

fact_sales = spark.sql("""
    select o.order_id
        , p.product_key
        , c.customer_key
        , o.order_date
        , o.quantity as order_quantity
        , o.total_amount as sale_amount
        , case when p.product_key is null then 'product_key_is_null'
                when c.customer_key is null then 'customer_key_is_null'
                else "is_valid"
        end as dq_note
        , current_timestamp() as load_ts
from ecommerce.e_comm_silver.orders as o
    left join ecommerce.e_comm_gold.dimproducts as p
    on o.product_id = p.product_id
    left join ecommerce.e_comm_gold.dimcustomers as c
    on o.customer_id = c.customer_id
    where o.dq_note="is_valid"                   
                       """)
display(fact_sales)                    
    

In [0]:
# Write to gold layer
fact_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce.e_comm_gold.factSales")

print(f"Total records in fact table: {fact_sales.count()}")
print("✅ Gold fact table ecommerce.e_comm_gold.factSales created successfully")